# Compare one image across variants

Pick one image and see how every preprocessing variant changed it, with its mask area and fallback flag.

Use this to check mask quality by eye. It reads the generated data only and never touches model results.


In [ ]:
SPLIT = "train"      # train, val, test. Use train or val while checking masks: test stays for the final run
CLASS = "ramen"      # ramen, pho, steak, filet_mignon, fried_rice, pad_thai,
                     # chocolate_cake, chocolate_mousse, caesar_salad, greek_salad
INDEX = 0            # n-th image of this split and class
IMAGE_ID = None      # or an exact id such as "1234567", which overrides INDEX
SAVE_FIGURE = False  # True writes a PNG to analyst/figures/


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_ROOT = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data")) / "preprocessing_data"
FIGURE_DIR = PROJECT_ROOT / "analyst" / "figures"

# Colab: uncomment after unzipping the variants
# DATA_ROOT = Path("/content/data/preprocessing_data")

VARIANTS = ["original", "clahe", "histeq",
            "gsam", "gsam_light", "sam",
            "gsam_crop", "gsam_light_crop", "sam_crop"]

available = [v for v in VARIANTS if (DATA_ROOT / f"{v}_data").is_dir()]
assert available, f"No variants in {DATA_ROOT}. Unzip them from data_zips/ on Drive into this folder."
print("available:", available)
print("missing:  ", [v for v in VARIANTS if v not in available])


In [ ]:
class_dir = DATA_ROOT / f"{available[0]}_data" / SPLIT / CLASS
assert class_dir.is_dir(), f"{class_dir} not found. Check SPLIT and CLASS."
ids = sorted(p.stem for p in class_dir.glob("*.jpg"))

image_id = IMAGE_ID if IMAGE_ID is not None else ids[INDEX]
assert image_id in ids, f"{image_id} is not in {SPLIT}/{CLASS}"
relpath = f"{SPLIT}/{CLASS}/{image_id}.jpg"
print(f"{relpath}  (image {ids.index(image_id) + 1} of {len(ids)})")


In [ ]:
# one row per segmentation method; fallback = 1 means the variant holds the original photo
rows = []
for mode in ["gsam", "gsam_light", "sam"]:
    path = DATA_ROOT / f"{mode}_stats.csv"
    if path.exists():
        stats = pd.read_csv(path).set_index("relpath")
        if relpath in stats.index:
            rows.append({"method": mode, "mask_area": stats.at[relpath, "mask_area"],
                         "fallback": stats.at[relpath, "fallback"]})
mask_info = pd.DataFrame(rows, columns=["method", "mask_area", "fallback"]).set_index("method")
mask_info


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10.5))
for ax, variant in zip(axes.flat, VARIANTS):
    path = DATA_ROOT / f"{variant}_data" / relpath
    ax.set_xticks([]), ax.set_yticks([])
    if not path.exists():
        ax.set_title(variant, color="grey")
        ax.text(0.5, 0.5, "not available", ha="center", va="center", color="grey", transform=ax.transAxes)
        continue
    img = Image.open(path)
    method = variant.removesuffix("_crop")
    is_fallback = method in mask_info.index and mask_info.at[method, "fallback"] == 1
    ax.imshow(img)
    ax.set_title(f"{variant}  {img.width}×{img.height}" + ("  FALLBACK" if is_fallback else ""),
                 color="crimson" if is_fallback else "black", fontsize=10)
fig.suptitle(relpath)
fig.tight_layout()

if SAVE_FIGURE:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURE_DIR / f"{SPLIT}_{CLASS}_{image_id}.png"
    fig.savefig(out, dpi=150)
    print("saved", out)
plt.show()
